In [1]:
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (FunctionTransformer, OneHotEncoder, SplineTransformer,
                                   StandardScaler)

import scripts.feature_engineering as feature
from scripts.evaluate import calculate_metrics

In [2]:
df = pd.read_csv('data/gold/train.csv')
df_cat = pd.read_csv('data/landing/item_categories.csv')
df_items = pd.read_csv('data/landing/items.csv')
df_shops = pd.read_csv('data/landing/shops.csv')
df_sales = pd.read_csv('data/landing/sales_train.csv')

df = feature.add_categories(df, df_cat, df_items)
df = feature.add_cities(df, df_shops)
df = feature.add_target_lags(df, (1, 2, 3, 4, 5, 6, 12))
df = feature.add_lag_stats(df)
df = feature.add_mean_lags(df)
df = feature.add_age_features(df)
df = feature.add_price_features(df, df_sales, df_items)
df = feature.add_calendar(df)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10884554 entries, 0 to 10884553
Data columns (total 34 columns):
 #   Column                Dtype  
---  ------                -----  
 0   date_block_num        int64  
 1   shop_id               int64  
 2   item_id               int64  
 3   target                float64
 4   item_category_id      uint8  
 5   category_type_id      uint8  
 6   item_type_id          uint8  
 7   city_id               uint8  
 8   target_lag_1          float32
 9   target_lag_2          float32
 10  target_lag_3          float32
 11  target_lag_4          float32
 12  target_lag_5          float32
 13  target_lag_6          float32
 14  target_lag_12         float32
 15  roll3                 float32
 16  roll6                 float32
 17  diff_1_2              float32
 18  item_mean_lag_1       float32
 19  shop_mean_lag_1       float32
 20  cat_mean_lag_1        float32
 21  shop_cat_mean_lag_1   float32
 22  city_item_mean_lag_1  float32
 23  pair_age        

In [3]:
ALPHA = 1.4429937238040738e-05
MIN_FREQ = 5     # магазины и категории реже 5 строк объединяются в «прочее»
N_KNOTS = 6      # сплайны на сильнейшие признаки
MAX_ITER = 1000

# drop_weak = True: лаги 3–6, цены и sin/cos месяца выкинуты
NUM_COLS = [
    'target_lag_1', 'target_lag_2', 'target_lag_12', 'roll3', 'roll6',
    'item_mean_lag_1', 'shop_mean_lag_1', 'cat_mean_lag_1', 'shop_cat_mean_lag_1', 'city_item_mean_lag_1',
    'pair_age', 'pair_since_last', 'item_age', 'diff_1_2', 'item_is_new',
]
SPLINE_COLS = ['target_lag_1', 'item_mean_lag_1', 'shop_cat_mean_lag_1', 'item_age']
ONEHOT_COLS = ['shop_id', 'item_category_id']
LINEAR_FEATURES = NUM_COLS + ONEHOT_COLS

TRAIN_SAMPLE: int | None = 2_000_000  # None = весь train

m = df['date_block_num']
train, valid = df[(m >= 12) & (m <= 31)], df[m == 32]
if TRAIN_SAMPLE is not None and len(train) > TRAIN_SAMPLE:
    train = train.sample(TRAIN_SAMPLE, random_state=42)

prep = ColumnTransformer(
    [
        # use_log = False: числовые признаки без логарифма, только NaN в 0 с флагом и стандартизация
        ('num', make_pipeline(
            SimpleImputer(strategy='constant', fill_value=0.0, add_indicator=True),
            StandardScaler(),
        ), NUM_COLS),
        # сплайн-базис: кусочно-кубическая функция вместо прямой на 4 главных признаках
        ('spline', make_pipeline(
            FunctionTransformer(np.log1p, feature_names_out='one-to-one'),
            SimpleImputer(strategy='constant', fill_value=0.0),
            SplineTransformer(n_knots=N_KNOTS, degree=3, extrapolation='constant'),
        ), SPLINE_COLS),
        ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=MIN_FREQ,
                                 dtype=np.float32), ONEHOT_COLS),
    ],
    sparse_threshold=1.0,  # one-hot разреженный, вся матрица остаётся разреженной
)

In [4]:
model = make_pipeline(prep, PoissonRegressor(alpha=ALPHA, max_iter=MAX_ITER))

t = time.perf_counter()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', ConvergenceWarning)
    model.fit(train[LINEAR_FEATURES], train['target'])
converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
print(f'{time.perf_counter() - t:.0f} с | сошлась: {converged} | итераций: {model[-1].n_iter_} '
      f'| признаков: {model[-1].coef_.size}')

res = calculate_metrics(
    model,
    splits={'train': (train[LINEAR_FEATURES], train['target']), 'valid': (valid[LINEAR_FEATURES], valid['target'])},
    model_name='Poisson',
    baselines={'zeros': np.zeros(len(valid)), 'naive': valid['target_lag_1'].fillna(0)},
)
res.round(4)

177 с | сошлась: True | итераций: 849 | признаков: 190


,rmse,mae,bias,r2,rmse_nonzero
train,0.8914,0.3163,-0.0008,0.4444,2.1914
valid,0.8982,0.3100,0.0113,0.4079,2.2577
valid: zeros,1.1982,0.2701,-0.2701,-0.0536,3.2177
valid: naive,1.0766,0.3202,0.0106,0.1494,2.6185


In [5]:
# Эффекты числовых признаков: прогноз умножается на multiplier при росте признака на 1 ст. отклонение.
# Сплайны и one-hot сюда не входят: их коэффициенты по одному не интерпретируются.
coef = pd.Series(model[-1].coef_, index=model[0].get_feature_names_out())
effect = pd.DataFrame({'coef': coef, 'multiplier': np.exp(coef)})
num = effect[effect.index.str.startswith('num__')]
num.reindex(num['coef'].abs().sort_values(ascending=False).index).head(15).round(3)

,coef,multiplier
num__item_age,-0.514,0.598
num__missingindicator_city_item_mean_lag_1,-0.314,0.730
num__pair_since_last,-0.308,0.735
num__pair_age,0.261,1.298
num__shop_mean_lag_1,-0.205,0.815
num__missingindicator_item_mean_lag_1,0.176,1.193
num__item_mean_lag_1,0.156,1.169
num__missingindicator_pair_age,-0.135,0.874
num__missingindicator_pair_since_last,-0.135,0.874
num__missingindicator_shop_cat_mean_lag_1,0.134,1.143
